# 03 · Check whether the curve generalizes
A single 80/20 test split is noisy. First fix that test split and evaluate degree 1, 2 and 3 using **five folds inside the training files only**. The already reported held-out test remains untouched while comparing flexibility. We keep degree 2 for a simple, interpretable polynomial unless a more complex fit brings a meaningful improvement.

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
DATA = ROOT / "data" / "processed" / "cycling_lactate.csv"
assert DATA.exists(), f"Dataset missing: {DATA}"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 20)
from sklearn.model_selection import GroupShuffleSplit,GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
df=pd.read_csv(DATA)
a,b=next(GroupShuffleSplit(test_size=.2,random_state=42).split(df,groups=df.athlete_id))
training=df.iloc[a].reset_index(drop=True)
held_out=df.iloc[b].reset_index(drop=True)
print(len(training),"training rows;",len(held_out),"untouched test rows")

## Five grouped folds
Every stage of a test file stays together. Report both the average validation error and its variation across folds.

In [ ]:
folds=GroupKFold(n_splits=5)
rows=[]
for degree in [1,2,3]:
    for fold,(fit_i,val_i) in enumerate(folds.split(training,groups=training.athlete_id),start=1):
        fit,val=training.iloc[fit_i],training.iloc[val_i]
        assert set(fit.athlete_id).isdisjoint(val.athlete_id)
        model=make_pipeline(PolynomialFeatures(degree,include_bias=False),LinearRegression())
        model.fit(fit[["training_load"]],fit.performance_score)
        mae=mean_absolute_error(val.performance_score,model.predict(val[["training_load"]]))
        rows.append({"degree":degree,"fold":fold,"MAE mmol/L":mae})
fold_results=pd.DataFrame(rows)
print(fold_results.pivot(index="fold",columns="degree",values="MAE mmol/L").round(3))
print(fold_results.groupby("degree")["MAE mmol/L"].agg(["mean","std"]).round(3))

## Compare with restraint
The quadratic improves substantially over the straight line. A cubic improves only marginally; that small difference is not evidence that an extra term is useful. Degree 2 remains the production curve. Report the held-out result from notebook 02 once, without reselecting based on it.

In [ ]:
means=fold_results.groupby("degree")["MAE mmol/L"].mean()
print(f"Quadratic improvement over linear: {means[1]-means[2]:.3f} mmol/L")
print(f"Cubic improvement over quadratic: {means[2]-means[3]:.3f} mmol/L")
print("Production choice: degree 2 polynomial curve fitting")

## Known limit
The original summary has no person ID (`name` is empty). A test-file group split blocks within-test leakage, but cannot guarantee that an individual has not appeared in both train and test under different file IDs. Stronger subject-level evaluation would require verified person identifiers.